In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

#for classification

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier




In [5]:
np.random.seed(42)
n_sutdent = 800
student_Data = {
'Studyhourperweek': np.random.randint(5,50,n_sutdent),
'AttendencePercentage' : np.random.randint(40,100,n_sutdent),
'PreviousMarks' : np.random.randint(30,95,n_sutdent),
'AssignmentScore' : np.random.randint (40,100,n_sutdent)
}

studyscaled = (np.array(student_Data['Studyhourperweek'])/50 ) * 100


finalscore = (
    studyscaled *0.25 +
    np.array(student_Data['AttendencePercentage'] )* 0.25  + 
    np.array(student_Data['PreviousMarks'] )* 0.25  + 
    np.array(student_Data['AssignmentScore'] )* 0.25  
) + np.random.normal(0,8, n_sutdent)

finalscore = np.clip(finalscore ,0,100)

def assigngrade(score):
    if score >= 75 :
        return 'A'
    elif score >= 65 :
        return 'B'
    elif score >= 55 :
        return 'C'
    elif score >= 45 :
        return 'D'
    else:
        return'F'
    
student_Data['Grade'] = [assigngrade(s) for s in finalscore]

dfstudentdata = pd.DataFrame(student_Data)
 

In [12]:
xstudent = dfstudentdata.drop('Grade',axis = 1)
ystudent = dfstudentdata['Grade']

In [14]:
xtrain , xtest , ytrain,ytest = train_test_split(xstudent,ystudent , test_size=0.2 , random_state=42,stratify= ystudent)

In [ ]:
rfgrademodel = RandomForestClassifier(n_estimators=100 , random_state= 42)
rfgrademodel.fit(xtrain,ytrain)

In [17]:
gradeprediction = rfgrademodel.predict(xtest)
gradeaccuracy = accuracy_score(ytest,gradeprediction)

print(gradeaccuracy)
print(gradeaccuracy*100)

0.4125
41.25


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# --- Option 1: Gradient Boosting (no extra install needed) ---
gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    xstudent, ystudent,
    test_size=0.2,
    random_state=42,
    stratify=ystudent
)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
print(f"Gradient Boosting: {accuracy_score(y_test, gb_pred)*100:.2f}%")

# --- Option 2: XGBoost (pip install xgboost) ---
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)
xgb_model.fit(X_train, y_train_enc)
xgb_pred = le.inverse_transform(xgb_model.predict(X_test))
print(f"XGBoost:           {accuracy_score(y_test, xgb_pred)*100:.2f}%")

# --- Detailed breakdown (use whichever model wins) ---
print(classification_report(y_test, gb_pred))

Gradient Boosting: 36.25%
XGBoost:           35.00%
              precision    recall  f1-score   support

           A       0.55      0.52      0.53        31
           B       0.30      0.29      0.30        41
           C       0.31      0.33      0.32        46
           D       0.34      0.40      0.37        30
           F       0.38      0.25      0.30        12

    accuracy                           0.36       160
   macro avg       0.38      0.36      0.36       160
weighted avg       0.37      0.36      0.36       160



C:\Users\Siddhu\AppData\Roaming\Python\Python313\site-packages\xgboost\training.py:200: UserWarning: [23:39:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1. DATA GENERATION
#    Fix applied: noise reduced from ±8 → ±3
# ─────────────────────────────────────────────
np.random.seed(42)
n_student = 800

student_data = {
    'StudyHourPerWeek':      np.random.randint(5,  50, n_student),
    'AttendancePercentage':  np.random.randint(40, 100, n_student),
    'PreviousMarks':         np.random.randint(30,  95, n_student),
    'AssignmentScore':       np.random.randint(40, 100, n_student),
}

study_scaled = (np.array(student_data['StudyHourPerWeek']) / 50) * 100

final_score = (
    study_scaled                                      * 0.25 +
    np.array(student_data['AttendancePercentage'])    * 0.25 +
    np.array(student_data['PreviousMarks'])           * 0.25 +
    np.array(student_data['AssignmentScore'])         * 0.25
) + np.random.normal(0, 3, n_student)   # ← reduced noise (was 8)

final_score = np.clip(final_score, 0, 100)

# ─────────────────────────────────────────────
# 2. GRADE ASSIGNMENT
#    Fix applied: wider bands (15-pt gaps, no D)
#    Old: F<45, D<55, C<65, B<75, A>=75
#    New: F<40, C<55, B<70, A>=70
# ─────────────────────────────────────────────
def assign_grade(score):
    if score >= 70:
        return 'A'
    elif score >= 55:
        return 'B'
    elif score >= 40:
        return 'C'
    else:
        return 'F'

student_data['FinalScore'] = final_score
student_data['Grade']      = [assign_grade(s) for s in final_score]

df = pd.DataFrame(student_data)

print("── Grade Distribution ──────────────────")
print(df['Grade'].value_counts().sort_index())
print()

# ─────────────────────────────────────────────
# 3. TRAIN / TEST SPLIT
# ─────────────────────────────────────────────
X = df.drop(columns=['Grade', 'FinalScore'])
y = df['Grade']
scores = df['FinalScore']

X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, scores,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ─────────────────────────────────────────────
# 4A. APPROACH 1 — Direct Classifier (baseline)
#     GradientBoostingClassifier on grade labels
# ─────────────────────────────────────────────
clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
clf.fit(X_train, y_train)
clf_pred = clf.predict(X_test)
clf_acc  = accuracy_score(y_test, clf_pred)

print("── Approach 1: Direct Classifier ───────")
print(f"Accuracy: {clf_acc*100:.2f}%")
print()

# ─────────────────────────────────────────────
# 4B. APPROACH 2 — Regressor → Grade (best)
#     Predict raw score, then threshold into grade
# ─────────────────────────────────────────────
reg = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
reg.fit(X_train, s_train)

predicted_scores = reg.predict(X_test)
reg_pred = [assign_grade(s) for s in predicted_scores]
reg_acc  = accuracy_score(y_test, reg_pred)
rmse     = np.sqrt(mean_squared_error(s_test, predicted_scores))

print("── Approach 2: Regressor → Grade ───────")
print(f"Accuracy : {reg_acc*100:.2f}%")
print(f"Score RMSE: {rmse:.2f} points")
print()

# ─────────────────────────────────────────────
# 5. DETAILED REPORT (best model)
# ─────────────────────────────────────────────
best_pred  = reg_pred if reg_acc >= clf_acc else clf_pred
best_label = "Regressor → Grade" if reg_acc >= clf_acc else "Direct Classifier"

print(f"── Best Model: {best_label} ─────────────")
print(classification_report(y_test, best_pred, zero_division=0))

# ─────────────────────────────────────────────
# 6. FEATURE IMPORTANCE (from regressor)
# ─────────────────────────────────────────────
print("── Feature Importances ─────────────────")
importances = pd.Series(reg.feature_importances_, index=X.columns)
for feat, imp in importances.sort_values(ascending=False).items():
    bar = '█' * int(imp * 40)
    print(f"  {feat:<25} {imp:.3f}  {bar}")
print()

# ─────────────────────────────────────────────
# 7. SUMMARY
# ─────────────────────────────────────────────
print("── Summary ─────────────────────────────")
print(f"  Direct Classifier accuracy : {clf_acc*100:.2f}%")
print(f"  Regressor → Grade accuracy : {reg_acc*100:.2f}%")
print(f"  Best approach              : {best_label}")

class session pratice

supervisied learning 
    classification 
    Regression
    

Binary Classification
